# HW-08: Named Entity Recognition из медицинских диалогов

**Трек C: Медицинские диалоги**  
Датасет: `omi-health/medical-dialogue-to-soap-summary`  
Модели: Mistral-7B-Instruct (4bit, fp16) + Qwen2.5-7B-Instruct (4bit)  
Железо: Apple M3 (MLX / Metal)

**Извлекаемые сущности:** `SYMPTOM`, `DIAGNOSIS`, `MEDICATION`, `DOSAGE`, `DURATION`, `SIDE_EFFECT`

In [1]:
%pip install -q mlx-lm datasets openai psutil pandas


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [1]:
import json
import time
import subprocess
import psutil
import os
import platform

import pandas as pd
from datasets import load_dataset
from openai import OpenAI

## 1. Локальное развёртывание моделей (MLX + OpenAI API)

Используем **MLX-LM** — фреймворк Apple для запуска LLM на Apple Silicon.
MLX работает поверх **Metal** (MPS) нативно, что даёт лучшую производительность чем PyTorch+MPS.

Сервер MLX поднимается в терминале и предоставляет **OpenAI-совместимый API** — работаем с ним через стандартный `openai` клиент.

### Сравниваемые конфигурации:

| # | Модель | Квантизация | RAM | Цель |
|---|--------|-------------|-----|------|
| 1 | `Mistral-7B-Instruct-v0.3-4bit` | 4-bit | ~4 GB | Quantized baseline |
| 2 | `Mistral-7B-Instruct-v0.3` | fp16 (full) | ~14 GB | Quantized vs full precision |
| 3 | `Qwen2.5-7B-Instruct-4bit` | 4-bit | ~4 GB | Разные архитектуры |

### Запуск сервера (в отдельном терминале):
```bash
# Модель 1 — Mistral 4bit
python -m mlx_lm server --model mlx-community/Mistral-7B-Instruct-v0.3-4bit --port 8080

# Модель 2 — Mistral fp16
python -m mlx_lm server --model mlx-community/Mistral-7B-Instruct-v0.3 --port 8080

# Модель 3 — Qwen 4bit
python -m mlx_lm server --model mlx-community/Qwen2.5-7B-Instruct-4bit --port 8080
```
Модели скачиваются автоматически при первом запуске.

In [24]:
# Конфигурация моделей
# fp16 идёт первой — запускаем пока памяти точно достаточно
MODELS = {
    "mistral-fp16":  "mlx-community/Mistral-7B-Instruct-v0.3",        # full precision, та же архитектура
    "mistral-4bit":  "mlx-community/Mistral-7B-Instruct-v0.3-4bit",   # quantized, baseline
    "qwen-4bit":     "mlx-community/Qwen2.5-7B-Instruct-4bit",        # quantized, другая архитектура
}

SERVER_PORT = 8080
SERVER_URL  = f"http://localhost:{SERVER_PORT}/v1"

# Клиент создаётся один раз — URL всегда один и тот же независимо от модели
client = OpenAI(base_url=SERVER_URL, api_key="local", timeout=120.0)

print("Модели для сравнения:")
for name, model_id in MODELS.items():
    print(f"  {name:15s} -> {model_id}")

Модели для сравнения:
  mistral-fp16    -> mlx-community/Mistral-7B-Instruct-v0.3
  mistral-4bit    -> mlx-community/Mistral-7B-Instruct-v0.3-4bit
  qwen-4bit       -> mlx-community/Qwen2.5-7B-Instruct-4bit


In [25]:
import signal

_server_process = None

def start_server(model_id: str, port: int = SERVER_PORT, timeout: int = 300) -> bool:
    """Запускает MLX сервер с указанной моделью и ждёт готовности."""
    global _server_process

    stop_server()

    # Убиваем все висящие MLX процессы (на случай краша ядра)
    subprocess.run(["pkill", "-f", "mlx_lm"], capture_output=True)
    time.sleep(2)

    print(f"Запускаем сервер: {model_id.split('/')[-1]} ...", end=" ", flush=True)
    _server_process = subprocess.Popen(
        ["python", "-m", "mlx_lm", "server", "--model", model_id, "--port", str(port)],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.PIPE,
    )

    deadline = time.time() + timeout
    while time.time() < deadline:
        if _server_process.poll() is not None:
            err = _server_process.stderr.read().decode()
            print(f"Процесс завершился с ошибкой:\n{err}")
            return False
        try:
            client.models.list()
            print("готов.")
            return True
        except Exception:
            time.sleep(2)

    print(f"Таймаут {timeout}s превышен.")
    stop_server()
    return False


def stop_server():
    """Останавливает текущий MLX сервер."""
    global _server_process
    if _server_process and _server_process.poll() is None:
        _server_process.terminate()
        _server_process.wait()
        print("Сервер остановлен.")
    _server_process = None


def benchmark_model(model_id: str, prompt: str = "List 3 common cold symptoms briefly.", max_tokens: int = 100) -> dict:
    """Замеряет latency и throughput модели на тестовом запросе."""
    start = time.perf_counter()
    mem_before = psutil.Process().memory_info().rss / 1024**2

    response = client.chat.completions.create(
        model=model_id,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=max_tokens,
    )

    elapsed = time.perf_counter() - start
    mem_after = psutil.Process().memory_info().rss / 1024**2
    out_tokens = response.usage.completion_tokens

    return {
        "model":        model_id.split("/")[-1],
        "latency_s":    round(elapsed, 2),
        "tokens":       out_tokens,
        "tok_per_sec":  round(out_tokens / elapsed, 1),
        "ram_delta_mb": round(mem_after - mem_before, 1),
        "response":     response.choices[0].message.content,
    }

In [26]:
# Прогоняем все модели последовательно — замеряем базовый throughput
benchmark_results = {}

for name, model_id in MODELS.items():
    print(f"\n{'='*50}")
    print(f"Модель: {name}")
    if start_server(model_id):
        result = benchmark_model(model_id)
        benchmark_results[name] = result
        print(f"  Latency:    {result['latency_s']} s")
        print(f"  Throughput: {result['tok_per_sec']} tok/s")
        print(f"  RAM delta:  {result['ram_delta_mb']} MB")
        stop_server()

# Сводная таблица
rows = [{
    'Модель':        name,
    'Latency (s)':   r['latency_s'],
    'tok/s':         r['tok_per_sec'],
    'RAM delta (MB)': r['ram_delta_mb'],
} for name, r in benchmark_results.items()]

pd.DataFrame(rows).set_index('Модель')


Модель: mistral-fp16
Запускаем сервер: Mistral-7B-Instruct-v0.3 ... готов.
  Latency:    12.52 s
  Throughput: 8.0 tok/s
  RAM delta:  0.2 MB
Сервер остановлен.

Модель: mistral-4bit
Запускаем сервер: Mistral-7B-Instruct-v0.3-4bit ... готов.
  Latency:    3.52 s
  Throughput: 28.4 tok/s
  RAM delta:  0.0 MB
Сервер остановлен.

Модель: qwen-4bit
Запускаем сервер: Qwen2.5-7B-Instruct-4bit ... готов.
  Latency:    1.44 s
  Throughput: 20.8 tok/s
  RAM delta:  0.0 MB
Сервер остановлен.


,Latency (s),tok/s,RAM delta (MB)
Модель,,,
mistral-fp16,12.52,8.0,0.2
mistral-4bit,3.52,28.4,0.0
qwen-4bit,1.44,20.8,0.0


### Анализ результатов развёртывания

| Модель | Latency | tok/s | Вывод |
|--------|---------|-------|-------|
| `mistral-fp16` | 12.52s | 8.0 | Full precision, максимальное качество |
| `mistral-4bit` | 3.52s | 28.4 | Quantized, в 3.6x быстрее fp16 |
| `qwen-4bit` | 1.44s | 20.8 | Другая архитектура, быстрая latency |

**Ключевые выводы:**

1. **Квантизация 4bit даёт 3.6x прирост throughput** — `mistral-4bit` генерирует 28.4 tok/s против 8.0 tok/s у `mistral-fp16` при той же архитектуре. Это прямое следствие меньшего объёма данных для загрузки с памяти на GPU.

2. **Latency у `qwen-4bit` ниже** (1.44s vs 3.52s), но это артефакт тестового промпта — Qwen сгенерировал меньше токенов на простом запросе. По tok/s Mistral-4bit быстрее (28.4 vs 20.8), что важнее для обработки больших объёмов.

3. **RAM delta близка к нулю** — `psutil` измеряет память Python-процесса ноутбука, а не MLX сервера. Реальное потребление: ~4GB для 4bit моделей, ~14GB для fp16 — измерено в разделе 3.5.

**Выбор для дальнейшей работы:** `mistral-4bit` — оптимальный баланс скорости и качества.

## 2. Подготовка данных

In [27]:
dataset = load_dataset("omi-health/medical-dialogue-to-soap-summary")
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['dialogue', 'soap', 'prompt', 'messages', 'messages_nosystem'],
        num_rows: 9250
    })
    validation: Dataset({
        features: ['dialogue', 'soap', 'prompt', 'messages', 'messages_nosystem'],
        num_rows: 500
    })
    test: Dataset({
        features: ['dialogue', 'soap', 'prompt', 'messages', 'messages_nosystem'],
        num_rows: 250
    })
})


In [28]:
# Структура датасета — смотрим поля
sample = dataset["train"][0]
print("Поля:", list(sample.keys()))

for key, value in sample.items():
    print(f"\n{'='*60}\n[{key.upper()}]")
    print(str(value)[:800])

Поля: ['dialogue', 'soap', 'prompt', 'messages', 'messages_nosystem']

[DIALOGUE]
Doctor: Hello, how can I help you today?
Patient: My son has been having some issues with speech and development. He's 13 years old now.
Doctor: I see. Can you tell me more about his symptoms? Does he have any issues with muscle tone or hypotonia?
Patient: No, he doesn't have hypotonia. But he has mild to moderate speech and developmental delay, and he's been diagnosed with attention deficit disorder.
Doctor: Thank you for sharing that information. We'll run some tests, including an MRI, to get a better understanding of your son's condition. 
(After the tests)
Doctor: The MRI results are in, and I'm glad to say that there are no structural brain anomalies. However, I did notice some physical characteristics. Does your son have any facial features like retrognathia, mild hypertelorism, or a

[SOAP]
S: The patient's mother reports that her 13-year-old son has mild to moderate speech and developmental delays

In [29]:
# Статистика
train_data = dataset["train"]
print(f"Размер train: {len(train_data)} примеров")

dialogue_col = next((c for c in ["dialogue", "conversation", "text", "input"] if c in train_data.column_names), None)
print(f"Колонка с диалогом: {dialogue_col}")

if dialogue_col:
    lengths = [len(str(x)) for x in train_data[dialogue_col][:500]]
    print(f"Длина диалогов (первые 500): min={min(lengths)}, max={max(lengths)}, avg={sum(lengths)//len(lengths)}")

Размер train: 9250 примеров
Колонка с диалогом: dialogue
Длина диалогов (первые 500): min=1359, max=3411, avg=2624


In [30]:
# Подвыборка
SAMPLE_SIZE = 1000
subset = train_data.select(range(SAMPLE_SIZE))

print(f"Подвыборка: {len(subset)} примеров")
print(f"Колонки: {subset.column_names}")

df = subset.to_pandas()
df.head(3)

Подвыборка: 1000 примеров
Колонки: ['dialogue', 'soap', 'prompt', 'messages', 'messages_nosystem']


,dialogue,soap,prompt,messages,messages_nosystem
0,"Doctor: Hello, how can I help you today?\nPati...",S: The patient's mother reports that her 13-ye...,Create a Medical SOAP note summary from the di...,"[{'role': 'system', 'content': 'You are an exp...","[{'role': 'user', 'content': 'You are an exper..."
1,"Doctor: Hello, what brings you in today?\nPati...","S: The patient, a 21-month-old male, presented...",Create a Medical SOAP note summary from the di...,"[{'role': 'system', 'content': 'You are an exp...","[{'role': 'user', 'content': 'You are an exper..."
2,"Doctor: Hello, how can I help you today?\nPati...","S: Patient reports experiencing fatigue, night...",Create a Medical SOAP note summary from the di...,"[{'role': 'system', 'content': 'You are an exp...","[{'role': 'user', 'content': 'You are an exper..."


### 2.1 Промпты для извлечения сущностей

Качество NER напрямую зависит от промпта. Используем:
- **System prompt** — объясняет задачу и формат вывода
- **Few-shot примеры** — 2 примера в промпте чтобы модель понимала формат
- **Структурированный JSON** — фиксированная схема для всех сущностей

Извлекаемые типы:

| Тип | Описание | Пример |
|-----|----------|--------|
| `SYMPTOM` | Жалобы и симптомы пациента | "chest pain", "fever" |
| `DIAGNOSIS` | Диагноз врача | "hypertension", "type 2 diabetes" |
| `MEDICATION` | Лекарственные препараты | "metformin", "aspirin" |
| `DOSAGE` | Дозировка препарата | "500mg", "twice daily" |
| `DURATION` | Продолжительность симптомов или лечения | "3 days", "2 weeks" |
| `SIDE_EFFECT` | Побочные эффекты | "nausea", "dizziness" |

In [31]:
# System prompt встраивается в первый user-запрос
# Mistral поддерживает ТОЛЬКО роли user/assistant — system вызывает ошибку
SYSTEM_PROMPT = """You are a medical NER (Named Entity Recognition) system.
Extract medical entities from doctor-patient dialogues and return them as JSON.

Entity types:
- SYMPTOM: patient complaints and symptoms (e.g. "chest pain", "fever for 3 days")
- DIAGNOSIS: medical diagnoses made by the doctor (e.g. "hypertension", "type 2 diabetes")
- MEDICATION: drug names mentioned (e.g. "metformin", "aspirin")
- DOSAGE: dosage or frequency of medication (e.g. "500mg", "twice daily", "10mg once a day")
- DURATION: duration of symptoms or treatment (e.g. "3 days", "2 weeks", "since last month")
- SIDE_EFFECT: side effects of medications (e.g. "nausea", "dizziness")

Rules:
- Return ONLY valid JSON, no explanation text
- Each entity type is a list of strings (can be empty list [])
- Extract exact phrases from the text, do not paraphrase
- If an entity is not mentioned, return empty list

Output format:
{
  "SYMPTOM": [...],
  "DIAGNOSIS": [...],
  "MEDICATION": [...],
  "DOSAGE": [...],
  "DURATION": [...],
  "SIDE_EFFECT": [...]
}"""

# Few-shot примеры (строго user/assistant)
FEW_SHOT_EXAMPLES = [
    {
        "role": "user",
        "content": """Patient: I've had a headache and fever for 2 days.
Doctor: Any nausea? I think this might be a viral infection. Take ibuprofen 400mg three times a day for 3 days.
Patient: I feel slightly nauseous too."""
    },
    {
        "role": "assistant",
        "content": """{
  "SYMPTOM": ["headache", "fever", "nausea"],
  "DIAGNOSIS": ["viral infection"],
  "MEDICATION": ["ibuprofen"],
  "DOSAGE": ["400mg three times a day"],
  "DURATION": ["2 days", "3 days"],
  "SIDE_EFFECT": []
}"""
    },
    {
        "role": "user",
        "content": """Patient: My blood pressure has been high and I feel dizzy.
Doctor: You have hypertension. I'll prescribe lisinopril 10mg once daily.
Patient: Will it cause side effects?
Doctor: Some patients experience a dry cough."""
    },
    {
        "role": "assistant",
        "content": """{
  "SYMPTOM": ["high blood pressure", "dizziness"],
  "DIAGNOSIS": ["hypertension"],
  "MEDICATION": ["lisinopril"],
  "DOSAGE": ["10mg once daily"],
  "DURATION": [],
  "SIDE_EFFECT": ["dry cough"]
}"""
    },
]


def build_messages(dialogue: str) -> list:
    """Собирает messages для OpenAI API.

    System prompt встраивается в первый user-запрос — совместимо с Mistral и Qwen.
    Структура: user(system+пример1) → assistant(ответ1) → user(пример2) → assistant(ответ2) → user(диалог)
    """
    first_user_content = SYSTEM_PROMPT + "\n\nHere is an example:\n" + FEW_SHOT_EXAMPLES[0]["content"]
    return (
        [{"role": "user", "content": first_user_content}]
        + [{"role": "assistant", "content": FEW_SHOT_EXAMPLES[1]["content"]}]
        + FEW_SHOT_EXAMPLES[2:]
        + [{"role": "user", "content": dialogue}]
    )


print(f"System prompt: {len(SYSTEM_PROMPT)} символов")
print(f"Few-shot примеров: {len(FEW_SHOT_EXAMPLES) // 2}")
msgs = build_messages("test")
print(f"Кол-во messages: {len(msgs)}")
print("Роли:", [m["role"] for m in msgs])

System prompt: 1021 символов
Few-shot примеров: 2
Кол-во messages: 5
Роли: ['user', 'assistant', 'user', 'assistant', 'user']


### 2.2 Парсинг ответа и обработка edge cases

Модели иногда возвращают невалидный JSON — нужна robust-обёртка:
- Markdown-блок ` ```json ... ``` ` вокруг ответа
- Лишний текст до/после JSON
- Пустой ответ
- Неверные типы значений (строка вместо списка)

In [32]:
import re

ENTITY_TYPES = ["SYMPTOM", "DIAGNOSIS", "MEDICATION", "DOSAGE", "DURATION", "SIDE_EFFECT"]
EMPTY_RESULT = {t: [] for t in ENTITY_TYPES}


def parse_ner_response(text: str) -> dict:
    """Парсит JSON из ответа модели. Обрабатывает edge cases."""
    if not text or not text.strip():
        return EMPTY_RESULT.copy()

    # Убираем EOS-токен Mistral и лишний текст после него
    text = text.split("</s>")[0]

    # Убираем markdown-блок ```json ... ```
    match = re.search(r"```(?:json)?\s*([\s\S]*?)```", text)
    if match:
        text = match.group(1)

    # Берём ПЕРВЫЙ JSON-объект из текста (игнорируем FOLLOW_UP и прочее)
    match = re.search(r'\{[\s\S]*?\}', text)
    if not match:
        return EMPTY_RESULT.copy()

    try:
        data = json.loads(match.group())
    except json.JSONDecodeError:
        return EMPTY_RESULT.copy()

    # Нормализуем: все типы должны быть списками строк
    result = {}
    for t in ENTITY_TYPES:
        val = data.get(t, [])
        if isinstance(val, str):
            val = [val] if val else []
        elif not isinstance(val, list):
            val = []
        result[t] = [str(v).strip() for v in val if v]

    return result


# Тесты edge cases
assert parse_ner_response("") == EMPTY_RESULT
assert parse_ner_response("Sorry, I cannot help") == EMPTY_RESULT
assert parse_ner_response('```json\n{"SYMPTOM": ["fever"]}\n```')["SYMPTOM"] == ["fever"]
assert parse_ner_response('{"SYMPTOM": "fever"}')["SYMPTOM"] == ["fever"]
# EOS токен
assert parse_ner_response('{"SYMPTOM": ["fever"]}</s>\n{"FOLLOW_UP": ["visit"]}')["SYMPTOM"] == ["fever"]
print("Все edge case тесты прошли.")

Все edge case тесты прошли.


### 2.3 Тест промпта на одном примере

In [33]:
# Тест на первом примере датасета
# (сервер должен быть запущен)

def extract_entities(dialogue: str, model_id: str, max_tokens: int = 512) -> dict:
    """Извлекает сущности из диалога через MLX сервер."""
    response = client.chat.completions.create(
        model=model_id,
        messages=build_messages(dialogue),
        max_tokens=max_tokens,
        temperature=0.0,  # детерминированный вывод для NER
    )
    return parse_ner_response(response.choices[0].message.content)


# Находим колонки
dialogue_col = next((c for c in ["dialogue", "conversation", "text", "input"] if c in df.columns), df.columns[0])
print(f"Колонка с диалогом: {dialogue_col}")
print(f"Колонки датасета: {list(df.columns)}")
print()
print("Пример диалога:")
print(df[dialogue_col].iloc[0][:500])

Колонка с диалогом: dialogue
Колонки датасета: ['dialogue', 'soap', 'prompt', 'messages', 'messages_nosystem']

Пример диалога:
Doctor: Hello, how can I help you today?
Patient: My son has been having some issues with speech and development. He's 13 years old now.
Doctor: I see. Can you tell me more about his symptoms? Does he have any issues with muscle tone or hypotonia?
Patient: No, he doesn't have hypotonia. But he has mild to moderate speech and developmental delay, and he's been diagnosed with attention deficit disorder.
Doctor: Thank you for sharing that information. We'll run some tests, including an MRI, to get 


In [35]:
# Запускаем тест — нужен активный сервер
active_model = list(MODELS.values())[1]  # mistral-4bit

if start_server(active_model):
    sample_dialogue = df[dialogue_col].iloc[0]
    
    # Сырой ответ модели
    response = client.chat.completions.create(
        model=active_model,
        messages=build_messages(sample_dialogue),
        max_tokens=512,
        temperature=0.0,
    )
    raw = response.choices[0].message.content
    print("Сырой ответ модели:")
    print(raw)
    print()
    
    # Парсинг
    result = parse_ner_response(raw)
    print("Извлечённые сущности:")
    for entity_type, values in result.items():
        print(f"  {entity_type}: {values}")
    
    stop_server()

Запускаем сервер: Mistral-7B-Instruct-v0.3-4bit ... готов.
Сырой ответ модели:
{
  "SYMPTOM": ["speech issues", "developmental delay", "attention deficit disorder"],
  "DIAGNOSIS": ["genetic disorder", "de novo frameshift variant in ZBTB18 gene"],
  "MEDICATION": [],
  "DOSAGE": [],
  "DURATION": [],
  "SIDE_EFFECT": []
}

For follow-up recommendations:

{
  "FOLLOW_UP": ["regular visits with a speech and language therapist", "occupational therapist", "psychologist", "regular check-ups with the doctor"]
}</s>

Извлечённые сущности:
  SYMPTOM: ['speech issues', 'developmental delay', 'attention deficit disorder']
  DIAGNOSIS: ['genetic disorder', 'de novo frameshift variant in ZBTB18 gene']
  MEDICATION: []
  DOSAGE: []
  DURATION: []
  SIDE_EFFECT: []
Сервер остановлен.


## 3. Оптимизация для IE

Реализуем техники оптимизации:

| Техника | Описание | Ожидаемый эффект | Статус |
|---------|----------|------------------|--------|
| **Короткий промпт** | Убираем few-shot, оставляем только system prompt | Меньше токенов на prefill → быстрее | ✅ Реализовано |
| **Batch processing** | Параллельные запросы через `ThreadPoolExecutor` | Выше суммарный throughput | ✅ Реализовано |
| **Квантизация** | Сравнение 4bit vs fp16 на одной архитектуре | Trade-off скорость/качество | ✅ Реализовано |
| **Prompt caching** | Кеш KV для повторяющейся части промпта | Меньше вычислений на prefill | ⚠️ MLX не поддерживает |
| **Flash Attention** | Оптимизированный алгоритм attention с O(N) памятью | Быстрее на длинных контекстах | ⚠️ MLX использует автоматически через Metal kernels |

### Об оптимизациях в MLX

**Flash Attention** — алгоритм вычисления attention с O(N) памятью вместо O(N²), который ускоряет inference на длинных контекстах. Напрямую через OpenAI API включить нельзя, но это не нужно:

MLX на Apple Silicon использует **оптимизированные Metal kernels** для матричных операций и attention — это функциональный эквивалент Flash Attention, реализованный на уровне фреймворка. Включается автоматически, явной настройки не требует.

Именно поэтому MLX на M3 даёт ~30-80 tok/s против ~10-15 tok/s у PyTorch+MPS при тех же моделях.

**В дальнейшем рассматриваем оптимизации:**
1. **Короткий промпт** — убираем few-shot, уменьшаем prefill
2. **Batch processing** — параллельные запросы, меньше простоев
3. **Сравнение квантизации** — 4bit vs fp16, trade-off скорость/качество

### 3.1 Базовый inference (sequential, без оптимизаций)

In [36]:
# Базовый последовательный inference — используем как baseline
# Берём небольшую подвыборку для быстрого сравнения
BENCH_SIZE = 50   # примеров для сравнения методов оптимизации
bench_dialogues = df[dialogue_col].iloc[:BENCH_SIZE].tolist()

def run_sequential(dialogues: list, model_id: str) -> dict:
    """Последовательный inference — один запрос за другим."""
    results = []
    latencies = []
    total_tokens = 0

    t_start = time.perf_counter()
    for dialogue in dialogues:
        t0 = time.perf_counter()
        response = client.chat.completions.create(
            model=model_id,
            messages=build_messages(dialogue),
            max_tokens=300,
            temperature=0.0,
        )
        latencies.append(time.perf_counter() - t0)
        total_tokens += response.usage.completion_tokens
        results.append(parse_ner_response(response.choices[0].message.content))

    total_time = time.perf_counter() - t_start
    return {
        "results": results,
        "total_time_s": round(total_time, 2),
        "avg_latency_s": round(sum(latencies) / len(latencies), 2),
        "throughput_tok_s": round(total_tokens / total_time, 1),
        "throughput_doc_s": round(len(dialogues) / total_time, 2),
        "total_tokens":  total_tokens,
    }

print(f'Подвыборка для benchmark: {BENCH_SIZE} диалогов')
print(f'Средняя длина диалога: {int(df[dialogue_col].iloc[:BENCH_SIZE].str.len().mean())} символов')

Подвыборка для benchmark: 50 диалогов
Средняя длина диалога: 2571 символов


### 3.2 Оптимизация 1: Prompt caching (короткий промпт без few-shot)

Few-shot примеры добавляют ~300 токенов к каждому запросу. Попробуем убрать их — это уменьшит prefill time.

In [37]:
def build_messages_short(dialogue: str) -> list:
    """Короткий промпт — только system инструкция, без few-shot примеров."""
    return [{
        "role": "user",
        "content": SYSTEM_PROMPT + "\n\nDialogue:\n" + dialogue
    }]


def run_sequential_short(dialogues: list, model_id: str) -> dict:
    """Последовательный inference с коротким промптом."""
    results = []
    latencies = []
    total_tokens = 0

    t_start = time.perf_counter()
    for dialogue in dialogues:
        t0 = time.perf_counter()
        try:
            response = client.chat.completions.create(
                model=model_id,
                messages=build_messages_short(dialogue),
                max_tokens=300,
                temperature=0.0,
            )
            latencies.append(time.perf_counter() - t0)
            total_tokens += response.usage.completion_tokens
            results.append(parse_ner_response(response.choices[0].message.content))
        except Exception as e:
            latencies.append(time.perf_counter() - t0)
            results.append(EMPTY_RESULT.copy())
            print(f"  Ошибка на диалоге {len(results)}: {e}")

    total_time = time.perf_counter() - t_start
    return {
        "results":        results,
        "total_time_s":   round(total_time, 2),
        "avg_latency_s":  round(sum(latencies) / len(latencies), 2),
        "throughput_tok_s": round(total_tokens / total_time, 1) if total_tokens else 0,
        "throughput_doc_s": round(len(dialogues) / total_time, 2),
        "total_tokens":   total_tokens,
    }

# Длина промптов
sample = bench_dialogues[0]
msgs_full  = build_messages(sample)
msgs_short = build_messages_short(sample)
full_chars  = sum(len(m["content"]) for m in msgs_full)
short_chars = sum(len(m["content"]) for m in msgs_short)
print(f"Полный промпт (few-shot): ~{full_chars} символов")
print(f"Короткий промпт:          ~{short_chars} символов")
print(f"Экономия:                 ~{full_chars - short_chars} символов ({100*(full_chars-short_chars)//full_chars}%)")

Полный промпт (few-shot): ~3962 символов
Короткий промпт:          ~3122 символов
Экономия:                 ~840 символов (21%)


### 3.3 Оптимизация 2: Batch processing (параллельные запросы)

MLX сервер однопоточный, но параллельные HTTP-запросы ставятся в очередь — это повышает утилизацию и сокращает общее время за счёт перекрытия latency.

In [38]:
from concurrent.futures import ThreadPoolExecutor, as_completed


def run_batch(dialogues: list, model_id: str, batch_size: int = 4) -> dict:
    """Batch processing — параллельные запросы через ThreadPoolExecutor.
    
    batch_size: сколько запросов отправляем одновременно.
    MLX сервер обрабатывает их последовательно, но HTTP overhead перекрывается.
    """
    results = [None] * len(dialogues)
    total_tokens = 0

    def fetch(idx: int, dialogue: str):
        response = client.chat.completions.create(
            model=model_id,
            messages=build_messages(dialogue),
            max_tokens=300,
            temperature=0.0,
        )
        return idx, response

    t_start = time.perf_counter()
    with ThreadPoolExecutor(max_workers=batch_size) as executor:
        futures = {executor.submit(fetch, i, d): i for i, d in enumerate(dialogues)}
        for future in as_completed(futures):
            idx, response = future.result()
            results[idx] = parse_ner_response(response.choices[0].message.content)
            total_tokens += response.usage.completion_tokens

    total_time = time.perf_counter() - t_start
    return {
        "results": results,
        "total_time_s": round(total_time, 2),
        "avg_latency_s": round(total_time / len(dialogues), 2),
        "throughput_tok_s": round(total_tokens / total_time, 1),
        "throughput_doc_s": round(len(dialogues) / total_time, 2),
        "total_tokens": total_tokens,
        "batch_size": batch_size,
    }

print('Функции batch processing определены.')

Функции batch processing определены.


### 3.4 Запуск всех вариантов и сравнение

In [21]:
# Запускаем все три варианта на одной модели
model_id = MODELS['mistral-4bit']
perf_results = {}

if start_server(model_id):
    print(f'Модель: {model_id.split("/")[-1]}')
    print(f'Диалогов: {BENCH_SIZE}\n')

    print('1/3 Sequential (full prompt)...')
    perf_results['sequential_full'] = run_sequential(bench_dialogues, model_id)
    print(f"  Время: {perf_results['sequential_full']['total_time_s']}s, "
          f"{perf_results['sequential_full']['throughput_doc_s']} doc/s, "
          f"{perf_results['sequential_full']['throughput_tok_s']} tok/s")

    print('2/3 Sequential (short prompt)...')
    perf_results['sequential_short'] = run_sequential_short(bench_dialogues, model_id)
    print(f"  Время: {perf_results['sequential_short']['total_time_s']}s, "
          f"{perf_results['sequential_short']['throughput_doc_s']} doc/s, "
          f"{perf_results['sequential_short']['throughput_tok_s']} tok/s")

    print('3/3 Batch (batch_size=4)...')
    perf_results['batch_4'] = run_batch(bench_dialogues, model_id, batch_size=4)
    print(f"  Время: {perf_results['batch_4']['total_time_s']}s, "
          f"{perf_results['batch_4']['throughput_doc_s']} doc/s, "
          f"{perf_results['batch_4']['throughput_tok_s']} tok/s")

    stop_server()

# Сводная таблица
rows = []
labels = {
    'sequential_full':  'Sequential + few-shot',
    'sequential_short': 'Sequential + short prompt',
    'batch_4':          'Batch (size=4) + few-shot',
}
for key, label in labels.items():
    r = perf_results[key]
    rows.append({
        'Вариант':       label,
        'Время (s)':     r['total_time_s'],
        'doc/s':         r['throughput_doc_s'],
        'tok/s':         r['throughput_tok_s'],
        'avg latency':   r['avg_latency_s'],
    })

pd.DataFrame(rows).set_index('Вариант')

Запускаем сервер: Mistral-7B-Instruct-v0.3-4bit ... готов.
Модель: Mistral-7B-Instruct-v0.3-4bit
Диалогов: 50

1/3 Sequential (full prompt)...
  Время: 341.35s, 0.15 doc/s, 21.1 tok/s
2/3 Sequential (short prompt)...
  Время: 307.39s, 0.16 doc/s, 20.8 tok/s
3/3 Batch (batch_size=4)...
  Время: 194.43s, 0.26 doc/s, 37.2 tok/s
Сервер остановлен.


,Время (s),doc/s,tok/s,avg latency
Вариант,,,,
Sequential + few-shot,341.35,0.15,21.1,6.83
Sequential + short prompt,307.39,0.16,20.8,6.15
Batch (size=4) + few-shot,194.43,0.26,37.2,3.89


### Анализ результатов оптимизации

| Вариант | Время (s) | doc/s | tok/s | avg latency |
|---------|-----------|-------|-------|-------------|
| Sequential + few-shot | 341.35 | 0.15 | 21.1 | 6.83s |
| Sequential + short prompt | 307.39 | 0.16 | 20.8 | 6.15s |
| Batch (size=4) + few-shot | **194.43** | **0.26** | **37.2** | **3.89s** |

**Ключевые выводы:**

1. **Batch processing даёт наибольший эффект** — общее время сократилось с 341s до 194s, то есть **на 43%**. Throughput вырос с 21.1 до 37.2 tok/s (+76%). Это самая эффективная из реализованных оптимизаций.

2. **Короткий промпт даёт скромный прирост** — 341s → 307s (**−10%**). Экономия на prefill (убираем ~300 токенов few-shot) частично нивелируется тем, что без примеров модель может генерировать более длинные и менее структурированные ответы. tok/s почти не изменился (21.1 → 20.8) — bottleneck не в prefill, а в генерации.

3. **Avg latency на запрос:** batch снижает её с 6.83s до 3.89s — за счёт перекрытия ожидания между запросами. Пока сервер генерирует ответ для одного запроса, следующие уже стоят в очереди.

4. **Prompt caching** (недоступен в MLX) дал бы дополнительное ускорение prefill — system prompt + few-shot (~500 токенов) вычислялся бы один раз для всей подвыборки.

### 3.5 Сравнение квантизации: 4bit vs fp16 vs Qwen-4bit

Прогоняем NER на `bench_dialogues` для каждой модели и сравниваем throughput, latency и потребление памяти.

In [22]:
quant_results = {}
QUANT_SIZE = 10  # небольшая подвыборка — fp16 медленная
quant_dialogues = bench_dialogues[:QUANT_SIZE]

for name, model_id in MODELS.items():
    print(f"\n{'='*50}")
    print(f"Модель: {name} ({model_id.split('/')[-1]})")
    if start_server(model_id):
        mem_before = psutil.virtual_memory().used / 1024**2
        stats = run_sequential(quant_dialogues, model_id)
        mem_after = psutil.virtual_memory().used / 1024**2
        stats['ram_used_mb'] = round(mem_after - mem_before, 1)
        quant_results[name] = stats
        print(f"  Время:      {stats['total_time_s']}s")
        print(f"  Throughput: {stats['throughput_tok_s']} tok/s, {stats['throughput_doc_s']} doc/s")
        print(f"  Avg latency:{stats['avg_latency_s']}s")
        print(f"  RAM delta:  {stats['ram_used_mb']} MB")
        stop_server()

# Сводная таблица
rows = []
for name, r in quant_results.items():
    rows.append({
        'Модель':        name,
        'Время (s)':     r['total_time_s'],
        'tok/s':         r['throughput_tok_s'],
        'doc/s':         r['throughput_doc_s'],
        'avg latency (s)': r['avg_latency_s'],
        'RAM delta (MB)': r['ram_used_mb'],
    })

pd.DataFrame(rows).set_index('Модель')


Модель: mistral-4bit (Mistral-7B-Instruct-v0.3-4bit)
Запускаем сервер: Mistral-7B-Instruct-v0.3-4bit ... готов.
  Время:      81.24s
  Throughput: 21.5 tok/s, 0.12 doc/s
  Avg latency:8.12s
  RAM delta:  2380.6 MB
Сервер остановлен.

Модель: mistral-fp16 (Mistral-7B-Instruct-v0.3)
Запускаем сервер: Mistral-7B-Instruct-v0.3 ... готов.
  Время:      173.47s
  Throughput: 7.8 tok/s, 0.06 doc/s
  Avg latency:17.35s
  RAM delta:  12914.2 MB
Сервер остановлен.

Модель: qwen-4bit (Qwen2.5-7B-Instruct-4bit)
Запускаем сервер: Qwen2.5-7B-Instruct-4bit ... готов.
  Время:      74.82s
  Throughput: 21.9 tok/s, 0.13 doc/s
  Avg latency:7.48s
  RAM delta:  1737.1 MB
Сервер остановлен.


,Время (s),tok/s,doc/s,avg latency (s),RAM delta (MB)
Модель,,,,,
mistral-4bit,81.24,21.5,0.12,8.12,2380.6
mistral-fp16,173.47,7.8,0.06,17.35,12914.2
qwen-4bit,74.82,21.9,0.13,7.48,1737.1


### Анализ сравнения квантизации

| Модель | Время (s) | tok/s | avg latency | RAM delta |
|--------|-----------|-------|-------------|----------|
| `mistral-4bit` | 81.24 | 21.5 | 8.12s | 2.3 GB |
| `mistral-fp16` | 173.47 | 7.8 | 17.35s | **12.6 GB** |
| `qwen-4bit` | **74.82** | **21.9** | **7.48s** | 1.7 GB |

**Ключевые выводы:**

1. **4bit vs fp16 (одна архитектура):** `mistral-fp16` в **2.1x медленнее** `mistral-4bit` (173s vs 81s) и потребляет **в 5.4x больше RAM** (12.6 GB vs 2.3 GB). При этом tok/s падает с 21.5 до 7.8 — почти в 3 раза. Квантизация до 4bit — очевидный выбор для продакшна на устройствах с ограниченной памятью.

2. **Qwen-4bit vs Mistral-4bit (разные архитектуры):** сопоставимая скорость (21.9 vs 21.5 tok/s), но Qwen потребляет меньше RAM (1.7 GB vs 2.3 GB). Qwen2.5 использует более эффективную архитектуру с GQA (grouped-query attention), что снижает потребление памяти при схожей производительности.

3. **RAM delta** — при загрузке модели в unified memory M3 система перераспределяет память, и `psutil.virtual_memory()` фиксирует это изменение. fp16 занимает ~12.6 GB, что близко к теоретическому значению для 7B модели в float16 (~14 GB).


## 4. Анализ производительности

Сравниваем модели по трём параметрам:

| Параметр | Метрика | Источник |
|-----|---------|----------|
| Скорость | tok/s, doc/s, avg latency | замеры из разделов 1 и 3 |
| Качество | Precision, Recall, F1 | сравнение с SOAP ground truth |
| Ресурсы | RAM delta (MB) | замеры из раздела 3.5 |

**Ground truth:** поле `soap` в датасете содержит структурированное резюме диалога. Извлекаем из него сущности тем же промптом и сравниваем с предсказаниями модели.

### 4.1 Подготовка ground truth из SOAP

SOAP-резюме содержит те же сущности что и диалог, но в сжатом структурированном виде. Используем его как эталон для оценки качества извлечения.

In [39]:
EVAL_SIZE = 50
eval_subset = train_data.select(range(EVAL_SIZE))
eval_df = eval_subset.to_pandas()

print(f'Eval подвыборка: {EVAL_SIZE} примеров')
print(f'Пример SOAP:\n{eval_df["soap"].iloc[0][:400]}')

Eval подвыборка: 50 примеров
Пример SOAP:
S: The patient's mother reports that her 13-year-old son has mild to moderate speech and developmental delays and has been diagnosed with attention deficit disorder. She denies any issues with muscle tone or hypotonia. The patient also exhibits certain physical characteristics, including retrognathia, mild hypertelorism, an elongated philtrum, thin upper lip, broad and short hands, mild syndactyly


### 4.2 Метрики качества NER

Для оценки NER используем **partial matching** — совпадение считается если предсказанная сущность содержится в эталонной или наоборот. Точное совпадение строк слишком строгое (модель может написать `"high blood pressure"` вместо `"hypertension"`).

- **Precision** = доля предсказанных сущностей, которые есть в ground truth
- **Recall** = доля сущностей из ground truth, которые нашла модель
- **F1** = среднее гармоническое precision и recall

In [40]:
def is_match(pred: str, gt: str) -> bool:
    """Partial match — одна строка содержится в другой (case-insensitive)."""
    pred, gt = pred.lower().strip(), gt.lower().strip()
    return pred in gt or gt in pred


def compute_metrics(predicted: dict, ground_truth: dict) -> dict:
    """Вычисляет precision, recall, F1 для одного примера."""
    tp = fp = fn = 0
    for entity_type in ENTITY_TYPES:
        preds = predicted.get(entity_type, [])
        gts   = ground_truth.get(entity_type, [])

        matched_gt = set()
        for p in preds:
            if any(is_match(p, g) for g in gts):
                tp += 1
            else:
                fp += 1

        for g in gts:
            if not any(is_match(p, g) for p in preds):
                fn += 1

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    return {'precision': round(precision, 3), 'recall': round(recall, 3), 'f1': round(f1, 3)}


# Тест
pred = {'SYMPTOM': ['fever', 'headache'], 'DIAGNOSIS': ['flu'], 'MEDICATION': [], 'DOSAGE': [], 'DURATION': [], 'SIDE_EFFECT': []}
gt   = {'SYMPTOM': ['high fever', 'headache'], 'DIAGNOSIS': ['influenza'], 'MEDICATION': [], 'DOSAGE': [], 'DURATION': [], 'SIDE_EFFECT': []}
print('Тест метрик:', compute_metrics(pred, gt))

Тест метрик: {'precision': 1.0, 'recall': 1.0, 'f1': 1.0}


### 4.3 Финальный прогон: NER на eval подвыборке (50 примеров)

Прогоняем каждую модель на 50 диалогах, извлекаем сущности и сравниваем с SOAP ground truth.

In [41]:
def run_eval(dialogues: list, soap_texts: list, model_id: str) -> dict:
    """Прогоняет NER и считает метрики качества против SOAP ground truth."""
    all_metrics = []
    total_tokens = 0
    latencies = []

    t_start = time.perf_counter()
    for dialogue, soap in zip(dialogues, soap_texts):
        t0 = time.perf_counter()
        try:
            response = client.chat.completions.create(
                model=model_id,
                messages=build_messages(dialogue),
                max_tokens=300,
                temperature=0.0,
            )
            pred = parse_ner_response(response.choices[0].message.content)
            total_tokens += response.usage.completion_tokens
        except Exception:
            pred = EMPTY_RESULT.copy()
        latencies.append(time.perf_counter() - t0)

        # Ground truth — извлекаем сущности из SOAP через тот же парсер
        # SOAP уже структурирован, ищем ключевые слова по секциям
        gt = extract_gt_from_soap(soap)
        all_metrics.append(compute_metrics(pred, gt))

    total_time = time.perf_counter() - t_start
    avg_p = sum(m['precision'] for m in all_metrics) / len(all_metrics)
    avg_r = sum(m['recall']    for m in all_metrics) / len(all_metrics)
    avg_f1= sum(m['f1']        for m in all_metrics) / len(all_metrics)

    return {
        'precision':      round(avg_p, 3),
        'recall':         round(avg_r, 3),
        'f1':             round(avg_f1, 3),
        'tok_per_sec':    round(total_tokens / total_time, 1),
        'avg_latency_s':  round(sum(latencies) / len(latencies), 2),
        'total_time_s':   round(total_time, 2),
    }


def extract_gt_from_soap(soap: str) -> dict:
    """Извлекает ground truth сущности из SOAP-резюме через LLM.
    
    Используем тот же промпт что и для диалога — SOAP содержит те же сущности
    но в более чистом виде, что даёт более точный ground truth.
    """
    try:
        response = client.chat.completions.create(
            model=list(MODELS.values())[0],  # mistral-4bit как эталонная модель
            messages=build_messages(soap),
            max_tokens=300,
            temperature=0.0,
        )
        return parse_ner_response(response.choices[0].message.content)
    except Exception:
        return EMPTY_RESULT.copy()


print('Функции eval определены.')

Функции eval определены.


### 4.4 Сводное сравнение всех моделей

In [42]:
# Генерируем ground truth из SOAP через batch processing
EVAL_DIALOGUES = eval_df['dialogue'].tolist()
EVAL_SOAPS     = eval_df['soap'].tolist()

gt_results = [None] * len(EVAL_SOAPS)
model_id = MODELS['mistral-4bit']

if start_server(model_id):
    print('Генерируем ground truth из SOAP (batch_size=4)...')

    def fetch_gt(idx, soap):
        try:
            response = client.chat.completions.create(
                model=model_id,
                messages=build_messages(soap),
                max_tokens=300,
                temperature=0.0,
            )
            return idx, parse_ner_response(response.choices[0].message.content)
        except Exception:
            return idx, EMPTY_RESULT.copy()

    with ThreadPoolExecutor(max_workers=4) as executor:
        futures = {executor.submit(fetch_gt, i, s): i for i, s in enumerate(EVAL_SOAPS)}
        for future in as_completed(futures):
            idx, result = future.result()
            gt_results[idx] = result

    print(f'GT готов: {len(gt_results)} примеров')
    stop_server()

print(f'Пример GT: {gt_results[0]}')

Запускаем сервер: Mistral-7B-Instruct-v0.3-4bit ... готов.
Генерируем ground truth из SOAP (batch_size=4)...
GT готов: 50 примеров
Сервер остановлен.
Пример GT: {'SYMPTOM': ['speech delays', 'developmental delays', 'attention deficit disorder'], 'DIAGNOSIS': ['genetic disorder', 'ZBTB18 mutation'], 'MEDICATION': [], 'DOSAGE': [], 'DURATION': [], 'SIDE_EFFECT': []}


In [43]:
print(f'Пример GT: {gt_results[1]}')

Пример GT: {'SYMPTOM': ['weakness in lower extremities', 'lumbar pain'], 'DIAGNOSIS': ['Spondylodiscitis with associated osteomyelitis'], 'MEDICATION': ['meropenem', 'vancomycin'], 'DOSAGE': ['100 mg/kg/day in three doses (meropenem)', '40 mg/kg/day in three doses (vancomycin)'], 'DURATION': [], 'SIDE_EFFECT': ['leukopenia with severe neutropenia']}


In [44]:
# Прогоняем каждую модель с batch processing (batch_size=4)
eval_results = {}

# Данные по RAM из раздела 3
ram_data = {
    'mistral-4bit': 2380.6,
    'mistral-fp16': 12914.2,
    'qwen-4bit':    1737.1,
}

for name, model_id in MODELS.items():
    print(f"\n{'='*50}")
    print(f'Модель: {name}')
    if start_server(model_id):
        predictions = [None] * len(EVAL_DIALOGUES)
        total_tokens = 0
        latencies = []

        def fetch(idx, dialogue):
            t0 = time.perf_counter()
            try:
                response = client.chat.completions.create(
                    model=model_id,
                    messages=build_messages(dialogue),
                    max_tokens=300,
                    temperature=0.0,
                )
                return idx, response, time.perf_counter() - t0
            except Exception:
                return idx, None, time.perf_counter() - t0

        t_start = time.perf_counter()
        with ThreadPoolExecutor(max_workers=4) as executor:
            futures = {executor.submit(fetch, i, d): i for i, d in enumerate(EVAL_DIALOGUES)}
            for future in as_completed(futures):
                idx, response, lat = future.result()
                latencies.append(lat)
                if response:
                    predictions[idx] = parse_ner_response(response.choices[0].message.content)
                    total_tokens += response.usage.completion_tokens
                else:
                    predictions[idx] = EMPTY_RESULT.copy()

        total_time = time.perf_counter() - t_start

        # Метрики качества
        metrics = [compute_metrics(p, g) for p, g in zip(predictions, gt_results)]
        avg_p  = round(sum(m['precision'] for m in metrics) / len(metrics), 3)
        avg_r  = round(sum(m['recall']    for m in metrics) / len(metrics), 3)
        avg_f1 = round(sum(m['f1']        for m in metrics) / len(metrics), 3)

        eval_results[name] = {
            'precision':     avg_p,
            'recall':        avg_r,
            'f1':            avg_f1,
            'tok_per_sec':   round(total_tokens / total_time, 1),
            'avg_latency_s': round(sum(latencies) / len(latencies), 2),
            'ram_mb':        ram_data[name],
        }
        print(f"  P={avg_p}, R={avg_r}, F1={avg_f1}")
        print(f"  tok/s={eval_results[name]['tok_per_sec']}, latency={eval_results[name]['avg_latency_s']}s")
        stop_server()

# Сводная таблица
rows = [{
    'Модель':         name,
    'Precision':      r['precision'],
    'Recall':         r['recall'],
    'F1':             r['f1'],
    'tok/s':          r['tok_per_sec'],
    'avg latency (s)':r['avg_latency_s'],
    'RAM (MB)':       r['ram_mb'],
} for name, r in eval_results.items()]

pd.DataFrame(rows).set_index('Модель')


Модель: mistral-fp16
Запускаем сервер: Mistral-7B-Instruct-v0.3 ... готов.
  P=0.517, R=0.631, F1=0.546
  tok/s=17.5, latency=31.13s
Сервер остановлен.

Модель: mistral-4bit
Запускаем сервер: Mistral-7B-Instruct-v0.3-4bit ... готов.
  P=0.487, R=0.615, F1=0.532
  tok/s=36.4, latency=15.39s
Сервер остановлен.

Модель: qwen-4bit
Запускаем сервер: Qwen2.5-7B-Instruct-4bit ... готов.
  P=0.345, R=0.593, F1=0.422
  tok/s=39.3, latency=13.4s
Сервер остановлен.


,Precision,Recall,F1,tok/s,avg latency (s),RAM (MB)
Модель,,,,,,
mistral-fp16,0.517,0.631,0.546,17.5,31.13,12914.2
mistral-4bit,0.487,0.615,0.532,36.4,15.39,2380.6
qwen-4bit,0.345,0.593,0.422,39.3,13.40,1737.1


### Анализ: сравнение трёх моделей

| Модель | Precision | Recall | F1 | tok/s | Latency | RAM |
|---|---|---|---|---|---|---|
| mistral-fp16 | 0.517 | 0.631 | **0.546** | 17.5 | 31.1s | 12.9 GB |
| mistral-4bit | 0.487 | 0.615 | **0.532** | 36.4 | 15.4s | 2.4 GB |
| qwen-4bit | 0.345 | 0.593 | **0.422** | 39.3 | 13.4s | 1.7 GB |

**Выводы:**

- **Mistral-fp16 vs Mistral-4bit**: разница F1 минимальна (+0.014), зато fp16 в **2× медленнее** (17.5 vs 36.4 tok/s) и требует в **5× больше RAM** (12.9 GB vs 2.4 GB). Для практических задач 4-битная квантизация — очевидный выбор: потери качества незначительны, выигрыш по скорости и памяти существенный.

- **Qwen-4bit**: заметно отстаёт по Precision (0.345 vs 0.487) и F1 (0.422 vs 0.532). При этом Recall близок к Mistral — модель извлекает много сущностей, но с меньшей точностью (шумит). Для медицинского NER это хуже, чем пропустить сущность. Qwen выигрывает по скорости и памяти, но не по качеству.

- **Recall > Precision** у всех моделей: модели склонны извлекать «лишние» сущности. Это типично для few-shot NER без fine-tuning — промпт задаёт направление, но не устраняет галлюцинации.

In [46]:
# Сравнение качества: few-shot vs без few-shot (mistral-4bit)
model_id = MODELS['mistral-4bit']
predictions_short = [None] * len(EVAL_DIALOGUES)

if start_server(model_id):
    total_tokens, latencies = 0, []
    def fetch_short(idx, dialogue):
        t0 = time.perf_counter()
        try:
            response = client.chat.completions.create(
                model=model_id,
                messages=build_messages_short(dialogue),
                max_tokens=300,
                temperature=0.0,
            )
            return idx, response, time.perf_counter() - t0
        except Exception:
            return idx, None, time.perf_counter() - t0

    t_start = time.perf_counter()
    with ThreadPoolExecutor(max_workers=4) as executor:
        futures = {executor.submit(fetch_short, i, d): i for i, d in enumerate(EVAL_DIALOGUES)}
        for future in as_completed(futures):
            idx, response, lat = future.result()
            latencies.append(lat)
            if response:
                predictions_short[idx] = parse_ner_response(response.choices[0].message.content)
                total_tokens += response.usage.completion_tokens
            else:
                predictions_short[idx] = EMPTY_RESULT.copy()
    total_time = time.perf_counter() - t_start
    stop_server()

    metrics_short = [compute_metrics(p, g) for p, g in zip(predictions_short, gt_results)]
    avg_p  = round(sum(m['precision'] for m in metrics_short) / len(metrics_short), 3)
    avg_r  = round(sum(m['recall']    for m in metrics_short) / len(metrics_short), 3)
    avg_f1 = round(sum(m['f1']        for m in metrics_short) / len(metrics_short), 3)

    # Сравнительная таблица few-shot vs no few-shot
    few_shot = eval_results.get('mistral-4bit', {})
    comparison = pd.DataFrame([
        {
            'Вариант':    'mistral-4bit + few-shot',
            'Precision':  few_shot.get('precision', '-'),
            'Recall':     few_shot.get('recall', '-'),
            'F1':         few_shot.get('f1', '-'),
            'tok/s':      few_shot.get('tok_per_sec', '-'),
        },
        {
            'Вариант':    'mistral-4bit + no few-shot',
            'Precision':  avg_p,
            'Recall':     avg_r,
            'F1':         avg_f1,
            'tok/s':      round(total_tokens / total_time, 1),
        },
    ]).set_index('Вариант')
    print(comparison)
    print(f"\nРазница F1: {round(avg_f1 - few_shot.get('f1', 0), 3):+.3f}")

Запускаем сервер: Mistral-7B-Instruct-v0.3-4bit ... готов.
Сервер остановлен.
                            Precision  Recall     F1  tok/s
Вариант                                                    
mistral-4bit + few-shot         0.487   0.615  0.532   36.4
mistral-4bit + no few-shot      0.416   0.587  0.475   34.5

Разница F1: -0.057


### Анализ: few-shot vs без few-shot

| Вариант | Precision | Recall | F1 | tok/s |
|---|---|---|---|---|
| mistral-4bit + few-shot | 0.487 | 0.615 | **0.532** | 36.4 |
| mistral-4bit + no few-shot | 0.416 | 0.587 | **0.475** | 34.5 |

**Выводы:**

- Few-shot примеры дают **+0.057 F1** — ощутимый прирост без изменения модели или инфраструктуры.
- Падение в основном по Precision (0.487 → 0.416): без примеров модель хуже понимает нужный формат и типы сущностей, извлекает «лишнее».
- Скорость практически не изменяется (36.4 vs 34.5 tok/s) — few-shot добавляет токены в промпт, но не влияет на скорость генерации.
- **Вывод**: несколько хорошо подобранных примеров в промпте — дешёвый и эффективный способ улучшить качество NER без fine-tuning.

### 4.5 Custom quantization: 2bit vs 4bit

Кастомная квантизация Mistral-7B в 2bit и 4bit и сравнение с community 4bit.

```bash
# 4bit (те же параметры что у community — для проверки воспроизводимости)
python -m mlx_lm convert \
    --hf-path mlx-community/Mistral-7B-Instruct-v0.3 \
    --mlx-path ./my-mistral-4bit \
    --quantize \
    --q-bits 4 \
    --q-group-size 64

# 2bit (агрессивная квантизация)
python -m mlx_lm convert \
    --hf-path mlx-community/Mistral-7B-Instruct-v0.3 \
    --mlx-path ./my-mistral-2bit \
    --quantize \
    --q-bits 2 \
    --q-group-size 64
```

In [ ]:
# Конфигурация кастомных моделей
CUSTOM_MODELS = {
    'my-mistral-4bit': './my-mistral-4bit',  # кастомная квантизация 4bit (group_size=64)
    'my-mistral-2bit': './my-mistral-2bit',  # кастомная квантизация 2bit
}

custom_results = {}
CUSTOM_SIZE = 20  # небольшая подвыборка
custom_dialogues = EVAL_DIALOGUES[:CUSTOM_SIZE]
custom_gt        = gt_results[:CUSTOM_SIZE]

for name, model_path in CUSTOM_MODELS.items():
    import os
    if not os.path.exists(model_path):
        print(f'Модель {name} не найдена — пропускаем. Запустите квантизацию выше.')
        continue

    print(f"\n{'='*50}")
    print(f'Модель: {name} ({model_path})')

    if start_server(model_path):
        predictions = [None] * CUSTOM_SIZE
        total_tokens, latencies = 0, []

        def fetch_custom(idx, dialogue):
            t0 = time.perf_counter()
            try:
                response = client.chat.completions.create(
                    model=model_path,
                    messages=build_messages(dialogue),
                    max_tokens=300,
                    temperature=0.0,
                )
                return idx, response, time.perf_counter() - t0
            except Exception:
                return idx, None, time.perf_counter() - t0

        t_start = time.perf_counter()
        with ThreadPoolExecutor(max_workers=4) as executor:
            futures = {executor.submit(fetch_custom, i, d): i for i, d in enumerate(custom_dialogues)}
            for future in as_completed(futures):
                idx, response, lat = future.result()
                latencies.append(lat)
                if response:
                    predictions[idx] = parse_ner_response(response.choices[0].message.content)
                    total_tokens += response.usage.completion_tokens
                else:
                    predictions[idx] = EMPTY_RESULT.copy()
        total_time = time.perf_counter() - t_start
        stop_server()

        metrics = [compute_metrics(p, g) for p, g in zip(predictions, custom_gt)]
        custom_results[name] = {
            'precision':     round(sum(m['precision'] for m in metrics) / len(metrics), 3),
            'recall':        round(sum(m['recall']    for m in metrics) / len(metrics), 3),
            'f1':            round(sum(m['f1']        for m in metrics) / len(metrics), 3),
            'tok_per_sec':   round(total_tokens / total_time, 1),
            'avg_latency_s': round(sum(latencies) / len(latencies), 2),
        }
        r = custom_results[name]
        print(f"  P={r['precision']}, R={r['recall']}, F1={r['f1']}")
        print(f"  tok/s={r['tok_per_sec']}, latency={r['avg_latency_s']}s")

# Сравнительная таблица с community 4bit
ref = eval_results.get('mistral-4bit', {})
rows = [{
    'Вариант':    'community 4bit (group_size=64)',
    'bits':       4,
    'group_size': 64,
    'Precision':  ref.get('precision', '-'),
    'Recall':     ref.get('recall', '-'),
    'F1':         ref.get('f1', '-'),
    'tok/s':      ref.get('tok_per_sec', '-'),
}]
for name, r in custom_results.items():
    bits = 4 if '4bit' in name else 2
    rows.append({
        'Вариант':    name,
        'bits':       bits,
        'group_size': 64,
        'Precision':  r['precision'],
        'Recall':     r['recall'],
        'F1':         r['f1'],
        'tok/s':      r['tok_per_sec'],
    })

pd.DataFrame(rows).set_index('Вариант')


Модель: my-mistral-4bit (./my-mistral-4bit)
Запускаем сервер: my-mistral-4bit ... готов.
Сервер остановлен.
  P=0.499, R=0.627, F1=0.547
  tok/s=29.9, latency=16.64s

Модель: my-mistral-2bit (./my-mistral-2bit)
Запускаем сервер: my-mistral-2bit ... готов.
Сервер остановлен.
  P=0.0, R=0.0, F1=0.0
  tok/s=45.4, latency=26.4s


,bits,group_size,Precision,Recall,F1,tok/s
Вариант,,,,,,
community 4bit (group_size=64),4,64,0.487,0.615,0.532,36.4
my-mistral-4bit,4,64,0.499,0.627,0.547,29.9
my-mistral-2bit,2,64,0.000,0.000,0.000,45.4


### Анализ: кастомная квантизация (2bit vs 4bit)

| Вариант | bits | group_size | Precision | Recall | F1 | tok/s |
|---|---|---|---|---|---|---|
| community 4bit | 4 | 64 | 0.487 | 0.615 | **0.532** | 36.4 |
| my-mistral-4bit | 4 | 64 | 0.499 | 0.627 | **0.547** | 29.9 |
| my-mistral-2bit | 2 | 64 | 0.0 | 0.0 | **0.000** | 45.4 |

**Выводы:**

- **my-mistral-4bit vs community 4bit**: кастомная квантизация показала чуть лучший F1 (0.547 vs 0.532). Оба варианта используют одинаковые параметры (4bit, group_size=64), небольшое расхождение объясняется разным порядком квантизации слоёв или реализацией affine-схемы. В целом результаты сопоставимы — качество воспроизводимо.

- **my-mistral-2bit**: модель деградировала до полного отказа от следования инструкциям — F1=0.0, ответы не содержат валидного JSON. 2-битная квантизация слишком агрессивно снижает точность весов: модель «забывает» формат вывода и структуру промпта. При этом скорость выросла до 45.4 tok/s — самая быстрая из всех вариантов, но практически бесполезная.

- **Вывод**: 4-битная квантизация — нижняя безопасная граница для инструкционных моделей на задаче NER. Переход на 2bit даёт ~25% прирост скорости ценой полной потери качества. Для production-использования: **4bit с group_size=64**.